## Step 1: Install and imports

In [ ]:
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn

## Step 2: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

## Step 3: Upload fused embeddings CSV

In [ ]:
uploaded = files.upload()
df = pd.read_csv("reef_fused_embeddings.csv")
print("Shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts().sort_index())

## Step 4: Dataset

In [ ]:
CNN_DIM = 256
NLP_DIM = 384

cnn_cols = [f"emb_{i}" for i in range(CNN_DIM)]
nlp_cols = [f"nlp_emb_{i+1}" for i in range(NLP_DIM)]

class FusionDataset(Dataset):
    def __init__(self, df):
        self.cnn = torch.tensor(df[cnn_cols].values, dtype=torch.float32)
        self.nlp = torch.tensor(df[nlp_cols].values, dtype=torch.float32)
        self.labels = torch.tensor(df["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.cnn[idx], self.nlp[idx], self.labels[idx]

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

train_ds = FusionDataset(train_df)
test_ds  = FusionDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f"Train: {len(train_ds)}  |  Test: {len(test_ds)}")

## Step 5: Attention-based fusion model

Projects CNN and NLP embeddings into a shared space, computes attention weights to learn how much to trust each modality per sample, then classifies.

In [ ]:
class AttentionFusionModel(nn.Module):
    def __init__(self, cnn_dim=256, nlp_dim=384, hidden_dim=256, num_classes=5, dropout=0.3):
        super().__init__()

        # Project each modality into shared hidden space
        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.nlp_proj = nn.Sequential(
            nn.Linear(nlp_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Attention: scalar score per modality -> softmax -> weighted sum
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2),   # 2 scores: one per modality
        )

        # Classifier on fused vector
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, cnn_emb, nlp_emb):
        cnn_h = self.cnn_proj(cnn_emb)   # (B, hidden_dim)
        nlp_h = self.nlp_proj(nlp_emb)   # (B, hidden_dim)

        # Compute attention weights
        combined = torch.cat([cnn_h, nlp_h], dim=-1)   # (B, hidden_dim*2)
        attn_scores = self.attention(combined)           # (B, 2)
        attn_weights = torch.softmax(attn_scores, dim=-1)  # (B, 2)

        # Weighted sum of the two modalities
        fused = (
            attn_weights[:, 0:1] * cnn_h +
            attn_weights[:, 1:2] * nlp_h
        )   # (B, hidden_dim)

        return self.classifier(fused), attn_weights

## Step 6: Training

Fixes applied:
- **Weighted loss** — down-weights overrepresented BAA-1 and BAA-4
- **Early stopping** — stops if val accuracy doesn't improve for 15 epochs
- **Best checkpoint** — saves the model at its highest val accuracy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Class weights (inverse frequency) ────────────────────────
label_counts = torch.tensor(
    [train_df["label"].value_counts().sort_index().values],
    dtype=torch.float32
).squeeze()
class_weights = (1.0 / label_counts)
class_weights = class_weights / class_weights.sum() * len(label_counts)
class_weights = class_weights.to(device)
print("Class weights:", class_weights.cpu().numpy().round(3))

model     = AttentionFusionModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Early stopping config ─────────────────────────────────────
EPOCHS        = 100
PATIENCE      = 15
best_val_acc  = 0.0
patience_ctr  = 0
best_state    = None
train_losses, val_accs = [], []

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for cnn_b, nlp_b, labels in train_loader:
        cnn_b, nlp_b, labels = cnn_b.to(device), nlp_b.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(cnn_b, nlp_b)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for cnn_b, nlp_b, labels in test_loader:
            cnn_b, nlp_b, labels = cnn_b.to(device), nlp_b.to(device), labels.to(device)
            logits, _ = model(cnn_b, nlp_b)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    acc = correct / total
    train_losses.append(total_loss / len(train_loader))
    val_accs.append(acc)

    # Save best
    if acc > best_val_acc:
        best_val_acc = acc
        patience_ctr = 0
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_losses[-1]:.4f} | Val Acc: {acc:.4f} | Best: {best_val_acc:.4f} | Patience: {patience_ctr}/{PATIENCE}")

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}. Best val acc: {best_val_acc:.4f}")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nRestored best model (val acc: {best_val_acc:.4f})")

## Step 7: Evaluate

In [ ]:
model.eval()
all_preds, all_labels, all_attn = [], [], []

with torch.no_grad():
    for cnn_b, nlp_b, labels in test_loader:
        cnn_b, nlp_b = cnn_b.to(device), nlp_b.to(device)
        logits, attn = model(cnn_b, nlp_b)
        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels)
        all_attn.append(attn.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_attn   = torch.cat(all_attn).numpy()

print("\n=== Classification Report ===")
print(classification_report(all_labels, all_preds,
      target_names=["BAA-0","BAA-1","BAA-2","BAA-3","BAA-4"]))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["BAA-0","BAA-1","BAA-2","BAA-3","BAA-4"],
            yticklabels=["BAA-0","BAA-1","BAA-2","BAA-3","BAA-4"])
plt.title("Confusion Matrix — Attention Fusion Model")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=150)
plt.show()

# Attention weights: how much CNN vs NLP on average
mean_attn = all_attn.mean(axis=0)
print(f"\nMean attention weights:")
print(f"  CNN modality: {mean_attn[0]:.3f}")
print(f"  NLP modality: {mean_attn[1]:.3f}")

## Step 8: Training curves

In [ ]:
best_epoch = int(np.argmax(val_accs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")

ax2.plot(val_accs, label="Val Accuracy")
ax2.axvline(best_epoch, color="red", linestyle="--", label=f"Best epoch {best_epoch+1} ({best_val_acc:.3f})")
ax2.set_title("Validation Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()
print(f"Best epoch: {best_epoch+1}  |  Best val acc: {best_val_acc:.4f}")

## Step 9: Save model

In [ ]:
torch.save(model.state_dict(), "/content/attention_fusion_model.pth")
print("Model saved.")

from google.colab import files as colab_files
colab_files.download("/content/attention_fusion_model.pth")
colab_files.download("/content/confusion_matrix.png")
colab_files.download("/content/training_curves.png")